In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

In [ ]:
# 2. Load Dataset
orig = pd.read_csv("/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv")
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/test.csv")
sample_submission = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv")

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)
print("Sample Submission Shape:", sample_submission.shape)

train.head()

In [ ]:
display(orig.info())
display(orig.describe())
display(orig.head())

print("Missing Values in Original Dataset:")
display(orig.isnull().sum().sort_values(ascending=False).head(20))

print("Duplicate Rows:", orig.duplicated().sum())

In [ ]:
orig['Compound'] = orig['Compound'].fillna('Unknown')
test['Compound'] = test['Compound'].fillna('Unknown')

In [ ]:
train = pd.concat([train, orig], axis=0).reset_index(drop=True)

In [ ]:
train = train.drop(columns=['Normalized_TyreLife', 'id'], axis=1)

In [ ]:
# 3. Dataset Overview

display(train.info())
display(train.describe())
display(train.head())

print("Missing Values in Train:")
display(train.isnull().sum().sort_values(ascending=False).head(20))

print("Duplicate Rows:", train.duplicated().sum())

In [ ]:
# 4. Target Analysis

TARGET = "PitNextLap"
ID = "id"

plt.figure(figsize=(6,4))
sns.countplot(x=train[TARGET])
plt.title("Target Distribution")
plt.show()

print(train[TARGET].value_counts(normalize=True))

In [ ]:
# 5. Basic EDA

num_cols = train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

num_cols = [c for c in num_cols if c not in [ID, TARGET]]

print("Numerical Columns:", len(num_cols))
print("Categorical Columns:", len(cat_cols))

In [ ]:
# Numerical Feature Distribution
"""
for col in num_cols[:10]:
    plt.figure(figsize=(6,4))
    sns.histplot(train[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.show()"""

In [ ]:
# Categorical Feature Countplot
"""
for col in cat_cols[:10]:
    plt.figure(figsize=(8,4))
    sns.countplot(y=train[col], order=train[col].value_counts().index[:15])
    plt.title(f"Countplot of {col}")
    plt.show()"""

In [ ]:
# Correlation Heatmap

corr_cols = num_cols + [TARGET]

plt.figure(figsize=(12,8))
sns.heatmap(train[corr_cols].corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# =====================================
# Advanced Feature Engineering
# Combined FE Version
# =====================================

def feature_engineering(df):

    df = df.copy()

    # =====================================
    # Basic Row Statistics
    # =====================================

    numeric_cols = df.select_dtypes(
        include=["int64", "float64"]
    ).columns.tolist()

    numeric_cols = [
        c for c in numeric_cols
        if c not in [ID, TARGET]
    ]

    if len(numeric_cols) > 0:

        df["num_mean"] = df[numeric_cols].mean(axis=1)

        df["num_std"] = df[numeric_cols].std(axis=1)

        df["num_min"] = df[numeric_cols].min(axis=1)

        df["num_max"] = df[numeric_cols].max(axis=1)

        df["num_range"] = (
            df["num_max"] - df["num_min"]
        )

    # =====================================
    # Tyre Features
    # =====================================

    if "TyreLife" in df.columns:

        df["TyreLife_sq"] = (
            df["TyreLife"] ** 2
        )

        df["TyreLife_cb"] = (
            df["TyreLife"] ** 3
        )

        df["TyreLife_log"] = np.log1p(
            df["TyreLife"]
        )

    # =====================================
    # Degradation Features
    # =====================================

    if (
        "Cumulative_Degradation" in df.columns and
        "TyreLife" in df.columns
    ):

        df["Deg_per_lap"] = (
            df["Cumulative_Degradation"] /
            (df["TyreLife"] + 1)
        )

        df["Deg_x_TyreLife"] = (
            df["Cumulative_Degradation"] *
            df["TyreLife"]
        )

        df["AbsDeg"] = (
            df["Cumulative_Degradation"].abs()
        )

    # =====================================
    # Race Progress Features
    # =====================================

    if "RaceProgress" in df.columns:

        df["Progress_sq"] = (
            df["RaceProgress"] ** 2
        )

        df["LapsRemaining"] = (
            1.0 - df["RaceProgress"]
        )

        df["IsEarlyRace"] = (
            df["RaceProgress"] < 0.25
        ).astype(int)

        df["IsMidRace"] = (
            (
                (df["RaceProgress"] >= 0.25) &
                (df["RaceProgress"] < 0.75)
            )
        ).astype(int)

        df["IsLateRace"] = (
            df["RaceProgress"] >= 0.75
        ).astype(int)

    # =====================================
    # Stint Features
    # =====================================

    if (
        "Stint" in df.columns and
        "TyreLife" in df.columns
    ):

        df["Stint_x_TyreLife"] = (
            df["Stint"] *
            df["TyreLife"]
        )

    if (
        "Stint" in df.columns and
        "RaceProgress" in df.columns
    ):

        df["Stint_x_Progress"] = (
            df["Stint"] *
            df["RaceProgress"]
        )

    # =====================================
    # Lap Features
    # =====================================

    if "LapNumber" in df.columns:

        df["LapNumber_sq"] = (
            df["LapNumber"] ** 2
        )

        df["LapNumber_log"] = np.log1p(
            df["LapNumber"]
        )

    if (
        "LapNumber" in df.columns and
        "Stint" in df.columns
    ):

        df["lap_per_stint"] = (
            df["LapNumber"] /
            (df["Stint"] + 1)
        )

    if (
        "LapNumber" in df.columns and
        "RaceProgress" in df.columns
    ):

        df["lap_progress_ratio"] = (
            df["LapNumber"] /
            (df["RaceProgress"] + 1e-6)
        )

    # =====================================
    # Tyre Ratio Features
    # =====================================

    if (
        "TyreLife" in df.columns and
        "LapNumber" in df.columns
    ):

        df["tyre_life_ratio"] = (
            df["TyreLife"] /
            (df["LapNumber"] + 1)
        )

    if (
        "tyre_life_ratio" in df.columns and
        "Cumulative_Degradation" in df.columns
    ):

        df["tyre_degradation_pressure"] = (
            df["tyre_life_ratio"] *
            df["Cumulative_Degradation"]
        )

    # =====================================
    # LapTime Features
    # =====================================

    if "LapTime (s)" in df.columns:

        df["LapTime_log"] = np.log1p(
            df["LapTime (s)"].clip(lower=0)
        )

    if "LapTime_Delta" in df.columns:

        df["DeltaAbs"] = (
            df["LapTime_Delta"].abs()
        )

        df["Delta_sq"] = (
            df["LapTime_Delta"] ** 2
        )

        df["lap_delta_sq"] = (
            df["LapTime_Delta"] ** 2
        )

    if (
        "LapTime (s)" in df.columns and
        "TyreLife" in df.columns
    ):

        df["LapTime_x_TyreLife"] = (
            df["LapTime (s)"] *
            df["TyreLife"]
        )

    if (
        "LapTime (s)" in df.columns and
        "Position" in df.columns
    ):

        df["LapTime_x_Position"] = (
            df["LapTime (s)"] *
            df["Position"]
        )

    if (
        "LapTime (s)" in df.columns and
        "Cumulative_Degradation" in df.columns
    ):

        df["laptime_degradation"] = (
            df["LapTime (s)"] *
            df["Cumulative_Degradation"]
        )

    # =====================================
    # Position Features
    # =====================================

    if "Position" in df.columns:

        df["Position_inv"] = (
            1.0 /
            (df["Position"] + 1)
        )

        df["IsLeading"] = (
            df["Position"] == 1
        ).astype(int)

        df["IsTop5"] = (
            df["Position"] <= 5
        ).astype(int)

        df["IsBack10"] = (
            df["Position"] >= 11
        ).astype(int)

        df["position_change"] = (
            df["Position"]
            .diff()
            .fillna(0)
        )

    if (
        "Position" in df.columns and
        "RaceProgress" in df.columns
    ):

        df["Position_x_Progress"] = (
            df["Position"] *
            df["RaceProgress"]
        )

    if (
        "Position" in df.columns and
        "LapNumber" in df.columns
    ):

        df["position_lap_interaction"] = (
            df["Position"] *
            df["LapNumber"]
        )

    if "Position_Change" in df.columns:

        df["PosChange_abs"] = (
            df["Position_Change"].abs()
        )

    if (
        "position_lap_interaction" in df.columns and
        "position_change" in df.columns
    ):

        df["position_race_dynamics"] = (
            df["position_lap_interaction"] +
            df["position_change"]
        )

    # =====================================
    # Cross Features
    # =====================================

    if (
        "TyreLife" in df.columns and
        "LapTime_Delta" in df.columns
    ):

        df["TyreLife_x_Delta"] = (
            df["TyreLife"] *
            df["LapTime_Delta"]
        )

    if (
        "RaceProgress" in df.columns and
        "TyreLife" in df.columns
    ):

        df["Progress_x_TyreLife"] = (
            df["RaceProgress"] *
            df["TyreLife"]
        )

    if (
        "RaceProgress" in df.columns and
        "Stint" in df.columns
    ):

        df["Progress_x_Stint"] = (
            df["RaceProgress"] *
            df["Stint"]
        )

    if (
        "Cumulative_Degradation" in df.columns and
        "RaceProgress" in df.columns
    ):

        df["Deg_x_Progress"] = (
            df["Cumulative_Degradation"] *
            df["RaceProgress"]
        )

    # =====================================
    # Compound Features
    # =====================================

    if "Compound" in df.columns:

        df["Compound_SOFT"] = (
            df["Compound"] == "SOFT"
        ).astype(int)

        df["Compound_HARD"] = (
            df["Compound"] == "HARD"
        ).astype(int)

        df["Compound_WET"] = (
            df["Compound"].isin(
                ["WET", "INTERMEDIATE"]
            )
        ).astype(int)

    # =====================================
    # Year Features
    # =====================================

    if "Year" in df.columns:

        for y in [2022, 2023, 2024, 2025]:

            df[f"Year_{y}"] = (
                df["Year"] == y
            ).astype(int)

    return df


# =====================================
# Apply Feature Engineering
# =====================================

train_fe = feature_engineering(train)

test_fe = feature_engineering(test)

print(train_fe.shape)
print(test_fe.shape)

In [ ]:
train_fe.columns

In [ ]:
# 7. Preprocessing

X = train_fe.drop(columns=[TARGET])
y = train_fe[TARGET]
X_test = test_fe.copy()

test_ids = X_test[ID]

#X = X.drop(columns=[ID])
X_test = X_test.drop(columns=[ID])

# Combine train and test for consistent encoding
combined = pd.concat([X, X_test], axis=0)

cat_cols = combined.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

for col in cat_cols:
    combined[col] = combined[col].astype(str).fillna("missing")
    le = LabelEncoder()
    combined[col] = le.fit_transform(combined[col])

# Fill missing numeric values
combined = combined.fillna(combined.median(numeric_only=True))

X = combined.iloc[:len(X), :]
X_test = combined.iloc[len(X):, :]

print("Final Train Shape:", X.shape)
print("Final Test Shape:", X_test.shape)

In [ ]:
# 8. Validation Strategy

N_SPLITS = 10
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=777)

In [ ]:
# 9. Model Training Function

import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import numpy as np

def train_model(model, X, y, X_test, model_name):

    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    scores = []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):

        print(f"\n===== {model_name} | Fold {fold+1} =====")

        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        # -----------------------------
        # LightGBM
        # -----------------------------
        if model_name == "LightGBM":

            model.fit(
                X_train,
                y_train,
                eval_set=[(X_valid, y_valid)],
                callbacks=[
                    lgb.early_stopping(200),
                    lgb.log_evaluation(200)
                ]
            )

        # -----------------------------
        # XGBoost
        # -----------------------------
        elif model_name == "XGBoost":

            model.fit(
                X_train,
                y_train,
                eval_set=[(X_valid, y_valid)],
                verbose=200
            )

        # -----------------------------
        # CatBoost
        # -----------------------------
        else:

            model.fit(
                X_train,
                y_train,
                eval_set=(X_valid, y_valid),
                use_best_model=True,
                verbose=200
            )

        # Predictions
        valid_pred = model.predict_proba(X_valid)[:, 1]
        test_pred = model.predict_proba(X_test)[:, 1]

        # Store OOF
        oof_preds[valid_idx] = valid_pred

        # Average Test Predictions
        test_preds += test_pred / N_SPLITS

        # Fold Score
        auc = roc_auc_score(y_valid, valid_pred)
        scores.append(auc)

        print(f"Fold {fold+1} ROC AUC: {auc:.5f}")

    # Final CV Score
    mean_auc = np.mean(scores)

    print(f"\n{model_name} Mean ROC AUC: {mean_auc:.5f}")

    return oof_preds, test_preds, mean_auc

In [ ]:
# 10. CatBoost Model
cat_model = CatBoostClassifier(
    iterations=5000,
    learning_rate=0.02,
    depth=8,
    l2_leaf_reg=5,
    bagging_temperature=1,
    random_strength=1.5,
    border_count=254,

    loss_function="Logloss",
    eval_metric="AUC",

    bootstrap_type="Bayesian",

    random_seed=777,
    verbose=200,

    task_type='GPU',
    use_best_model=True
)

cat_oof, cat_test, cat_score = train_model(
    cat_model, X, y, X_test, "CatBoost"
)

In [ ]:
# 11. LightGBM Model

lgbm_model = LGBMClassifier(
    n_estimators=5000,
    learning_rate=0.02,

    max_depth=-1,
    num_leaves=63,

    min_child_samples=40,
    min_split_gain=0.01,

    subsample=0.85,
    subsample_freq=1,

    colsample_bytree=0.75,

    reg_alpha=0.5,
    reg_lambda=1.5,

    objective="binary",
    metric="auc",

    random_state=777,
    device_type='gpu',
    early_stopping_rounds=200
)
lgbm_oof, lgbm_test, lgbm_score = train_model(
    lgbm_model, X, y, X_test, "LightGBM"
)

In [ ]:
# 12. XGBoost Model
xgb_model = XGBClassifier(
    n_estimators=5000,
    learning_rate=0.02,

    max_depth=8,
    min_child_weight=3,

    subsample=0.85,
    colsample_bytree=0.75,

    gamma=0.1,

    reg_alpha=0.5,
    reg_lambda=2,

    eval_metric="auc",

    random_state=777,

    tree_method="hist",
    device='cuda',
    early_stopping_rounds=200
)
xgb_oof, xgb_test, xgb_score = train_model(
    xgb_model, X, y, X_test, "XGBoost"
)

In [ ]:
# 13. Model Comparison

model_scores = pd.DataFrame({
    "Model": ["CatBoost", "LightGBM", "XGBoost"],
    "CV ROC AUC": [cat_score, lgbm_score, xgb_score]
})

model_scores.sort_values("CV ROC AUC", ascending=False)

In [ ]:
# 14. Ensemble
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score

# -----------------------------
# OOF Predictions Matrix
# -----------------------------
oof_matrix = np.vstack([
    cat_oof,
    lgbm_oof,
    xgb_oof
]).T

# -----------------------------
# Objective Function
# -----------------------------
def objective(weights):

    weights = np.array(weights)

    ensemble_pred = np.dot(oof_matrix, weights)

    score = roc_auc_score(y, ensemble_pred)

    return -score   # minimize negative AUC


# -----------------------------
# Initial Weights
# -----------------------------
initial_weights = [0.33, 0.33, 0.34]

# -----------------------------
# Constraints
# -----------------------------
constraints = {
    'type': 'eq',
    'fun': lambda w: 1 - sum(w)
}

bounds = [(0, 1)] * 3

# -----------------------------
# Optimize
# -----------------------------
result = minimize(
    objective,
    initial_weights,
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

# -----------------------------
# Best Weights
# -----------------------------
best_weights = result.x

print("\nBest Weights:")
print(f"CatBoost : {best_weights[0]:.4f}")
print(f"LightGBM : {best_weights[1]:.4f}")
print(f"XGBoost  : {best_weights[2]:.4f}")

# -----------------------------
# Final Ensemble
# -----------------------------
ensemble_oof = np.dot(oof_matrix, best_weights)

ensemble_test = (
    best_weights[0] * cat_test +
    best_weights[1] * lgbm_test +
    best_weights[2] * xgb_test
)

# -----------------------------
# Final Score
# -----------------------------
ensemble_score = roc_auc_score(y, ensemble_oof)

print(f"\nOptimized Ensemble ROC AUC: {ensemble_score:.6f}")

In [ ]:
# 15. Feature Importance using LightGBM

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": lgbm_model.feature_importances_
}).sort_values("Importance", ascending=False)

plt.figure(figsize=(10,8))
sns.barplot(
    data=feature_importance.head(20),
    x="Importance",
    y="Feature"
)
plt.title("Top 20 Feature Importance")
plt.show()

feature_importance.head(20)

In [ ]:
# 16. Create Submission File

submission = pd.DataFrame({
    ID: test_ids,
    TARGET: ensemble_test
})

submission.to_csv("submission.csv", index=False)

submission.head()

In [ ]:
# 17. Conclusion

print("Final CV Scores")
print(model_scores)
print("\nEnsemble CV ROC AUC:", ensemble_score)
print("\nSubmission file created: submission.csv")